In [ ]:
# ==========================================
# ANALYSE DES PRÉDICTIONS DE VICTOIRES POKÉMON
# ==========================================

# 1. IMPORTATION DES BIBLIOTHÈQUES

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ==========================================
# 2. CHARGEMENT DES DONNÉES
# ==========================================

pokemon = pd.read_csv("pokemon.csv")
combats = pd.read_csv("combats.csv")

print("Dimensions Pokemon :", pokemon.shape)
print("Dimensions Combats :", combats.shape)

# ==========================================
# 3. NETTOYAGE DES DONNÉES
# ==========================================

# Compléter le nom manquant du Pokémon #62
pokemon.loc[pokemon["#"] == 62, "Name"] = "Primeape"

# Remplacer les valeurs manquantes de Type 2
pokemon["Type 2"] = pokemon["Type 2"].fillna("None")

print("\nValeurs manquantes :")
print(pokemon.isnull().sum())

# ==========================================
# 4. CALCUL DU POURCENTAGE DE VICTOIRE
# ==========================================

# Nombre total de combats par Pokémon
total_battles = pd.concat([
    combats["First_pokemon"],
    combats["Second_pokemon"]
]).value_counts()

# Nombre de victoires
wins = combats["Winner"].value_counts()

# Création du DataFrame
win_rate = pd.DataFrame({
    "Total_Battles": total_battles,
    "Wins": wins
}).fillna(0)

# Calcul du pourcentage de victoire
win_rate["Win_Percentage"] = (
    win_rate["Wins"] / win_rate["Total_Battles"]
) * 100

# ==========================================
# 5. FUSION DES DONNÉES
# ==========================================

pokemon = pokemon.merge(
    win_rate["Win_Percentage"],
    left_on="#",
    right_index=True,
    how="left"
)

pokemon["Win_Percentage"] = pokemon["Win_Percentage"].fillna(0)

print("\nAperçu des données :")
print(pokemon.head())

# ==========================================
# 6. MATRICE DE CORRÉLATION
# ==========================================

corr_columns = [
    "HP",
    "Attack",
    "Defense",
    "Sp. Atk",
    "Sp. Def",
    "Speed",
    "Win_Percentage"
]

plt.figure(figsize=(10,6))

sns.heatmap(
    pokemon[corr_columns].corr(),
    annot=True,
    cmap="coolwarm"
)

plt.title("Matrice de corrélation")
plt.show()

# ==========================================
# 7. PAIRPLOT
# ==========================================

sns.pairplot(
    pokemon[
        ["HP", "Attack", "Defense", "Speed", "Win_Percentage"]
    ]
)

plt.show()

# ==========================================
# 8. TOP 10 POKÉMON
# ==========================================

top10 = pokemon.sort_values(
    by="Win_Percentage",
    ascending=False
).head(10)

print("\nTop 10 Pokémon :")
print(
    top10[
        ["Name",
         "HP",
         "Attack",
         "Defense",
         "Speed",
         "Win_Percentage"]
    ]
)

plt.figure(figsize=(10,5))

sns.barplot(
    data=top10,
    x="Win_Percentage",
    y="Name",
    palette="viridis"
)

plt.title("Top 10 Pokémon selon le pourcentage de victoire")
plt.show()

# ==========================================
# 9. PRÉPARATION MACHINE LEARNING
# ==========================================

features = [
    "HP",
    "Attack",
    "Defense",
    "Sp. Atk",
    "Sp. Def",
    "Speed"
]

X = pokemon[features]
y = pokemon["Win_Percentage"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# ==========================================
# 10. RÉGRESSION LINÉAIRE
# ==========================================

lr = LinearRegression()

lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(
    y_test,
    pred_lr
)

print("\nMAE Régression Linéaire :", round(mae_lr, 2))

# ==========================================
# 11. ARBRE DE DÉCISION
# ==========================================

dt = DecisionTreeRegressor(
    random_state=42
)

dt.fit(X_train, y_train)

pred_dt = dt.predict(X_test)

mae_dt = mean_absolute_error(
    y_test,
    pred_dt
)

print("MAE Arbre de Décision :", round(mae_dt, 2))

# ==========================================
# 12. FORÊT ALÉATOIRE
# ==========================================

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(
    y_test,
    pred_rf
)

print("MAE Forêt Aléatoire :", round(mae_rf, 2))

# ==========================================
# 13. COMPARAISON DES MODÈLES
# ==========================================

results = pd.DataFrame({
    "Modèle": [
        "Régression Linéaire",
        "Arbre de Décision",
        "Forêt Aléatoire"
    ],
    "MAE": [
        mae_lr,
        mae_dt,
        mae_rf
    ]
})

results = results.sort_values(
    by="MAE"
)

print("\nComparaison des modèles")
print(results)

plt.figure(figsize=(8,4))

sns.barplot(
    data=results,
    x="Modèle",
    y="MAE",
    palette="Set2"
)

plt.title("Comparaison des modèles (MAE)")
plt.show()

# ==========================================
# 14. PCA (ACP)
# ==========================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

print("\nVariance expliquée :")
print(pca.explained_variance_ratio_)

plt.figure(figsize=(8,6))

scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=y,
    cmap="viridis"
)

plt.colorbar(scatter)

plt.xlabel("Composante principale 1")
plt.ylabel("Composante principale 2")
plt.title("Projection PCA des Pokémon")

plt.show()

# ==========================================
# 15. CONCLUSION
# ==========================================

best_model = results.iloc[0]

print("\nMeilleur modèle :")
print(best_model["Modèle"])
print("MAE :", round(best_model["MAE"], 2))